# Phase 2: Isolation Forest Novelty Engine (20 Marks)

**Team Astra** | NSSC 2026 | IIT Kharagpur  
**Lead:** 

---

## Objectives
1. Feed 1D latent vectors into an Isolation Forest for Novelty Scoring
2. Invert native IF scores so that **higher = more anomalous**
3. Fuse metadata (sun_angle, season) with latent embeddings
4. Apply **three independent** statistical thresholding methods:
   - Generalized Pareto Distribution (Extreme Value Theory)
   - KDE Knee Detection
   - Median Absolute Deviation (Modified Z-score)
5. Determine anomalies via consensus voting (≥2 of 3 methods)

### Key Constraints
- **No arbitrary fixed-count cutoffs**
- **No contamination parameter** to decide anomaly count
- **Mathematical justification** required for threshold boundary

In [ ]:
# ─── System Setup ─────────────────────────────────────────────────────
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

IMAGE_DIR = os.path.join(PROJECT_ROOT, 'data', 'images')
METADATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'source_image_metadata.csv')
LATENT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'latent_vectors')
PLOTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'plots')
SCORES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'scores')

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import set_seed
from src.latent_utils import load_latents, plot_tsne, plot_umap
from src.isolation_forest import (
    build_features,
    run_isolation_forest,
    compute_threshold_gpd,
    compute_threshold_kde_knee,
    compute_threshold_mad,
    consensus_threshold,
    plot_score_distribution,
    save_scores,
)

set_seed(42)
print('Phase 2 imports ready.')

## 2.1 Load Latent Vectors

Load the latent vectors extracted in Phase 1 from the best model version.

In [ ]:
# Load latent vectors from the best version (v5)
# Also load v1 for comparison
BEST_VERSION = 'v5'

latents, filenames = load_latents(save_dir=LATENT_DIR, version=BEST_VERSION)
print(f'Latent vectors shape: {latents.shape}')
print(f'Number of images: {len(filenames)}')
print(f'Latent dimension: {latents.shape[1]}')

## 2.2 Metadata Fusion

Concatenate the 1D image embedding with an encoded metadata vector:
- Normalized `sun_angle` (min-max → [0, 1])
- Cyclically encoded `season` (sin, cos)

In [ ]:
# Build combined feature vectors
metadata_path = METADATA_PATH if os.path.exists(METADATA_PATH) else None

features, info_df = build_features(
    latents=latents,
    filenames=filenames,
    metadata_path=metadata_path,
)

print(f'\nCombined feature shape: {features.shape}')
print(f'Info DataFrame columns: {list(info_df.columns)}')
info_df.head()

## 2.3 Isolation Forest Scoring

Fit Isolation Forest on the fused feature vectors. The native `decision_function` scores are **inverted** so that higher score = greater anomalousness.

In [ ]:
# Run Isolation Forest
novelty_scores = run_isolation_forest(
    features,
    n_estimators=300,
    max_samples='auto',
    random_state=42,
)

print(f'\nScore Statistics:')
print(f'  Min:    {novelty_scores.min():.6f}')
print(f'  Max:    {novelty_scores.max():.6f}')
print(f'  Mean:   {novelty_scores.mean():.6f}')
print(f'  Median: {np.median(novelty_scores):.6f}')
print(f'  Std:    {novelty_scores.std():.6f}')

## 2.4 Statistical Thresholding

### Method 1: Generalized Pareto Distribution (Extreme Value Theory)

Under the **Pickands–Balkema–De Haan theorem**, exceedances over a sufficiently high threshold follow a Generalized Pareto Distribution. We fit GPD to scores above the 90th percentile and compute the quantile at p < 0.01.

In [ ]:
# Method 1: GPD (Extreme Value Theory)
threshold_gpd, info_gpd = compute_threshold_gpd(
    novelty_scores, tail_percentile=90.0, p_value=0.01
)

print(f'\nGPD Threshold Details:')
for k, v in info_gpd.items():
    print(f'  {k}: {v}')

### Method 2: KDE Knee Detection

Estimate the probability density of novelty scores via Gaussian KDE. The **knee point** where density drops sharply marks the transition from the dense "normal" region to the sparse "anomalous" tail. Detected via second-derivative (curvature) analysis of the log-density.

In [ ]:
# Method 2: KDE Knee Detection
threshold_kde, info_kde = compute_threshold_kde_knee(novelty_scores, n_points=1000)

print(f'\nKDE Knee Threshold Details:')
for k, v in info_kde.items():
    print(f'  {k}: {v}')

### Method 3: Median Absolute Deviation (MAD)

The MAD is a **robust measure of dispersion** resistant to outliers. An observation is anomalous if its **Modified Z-score** exceeds 3.5 (per Iglewicz & Hoaglin, 1993):

$$\text{Modified Z-score} = \frac{0.6745 \cdot (x_i - \tilde{x})}{\text{MAD}}$$

Equivalently: $\text{threshold} = \tilde{x} + \frac{k \cdot \text{MAD}}{0.6745}$ where $k = 3.5$.

In [ ]:
# Method 3: MAD (Modified Z-score)
threshold_mad, info_mad = compute_threshold_mad(novelty_scores, k=3.5)

print(f'\nMAD Threshold Details:')
for k, v in info_mad.items():
    print(f'  {k}: {v}')

## 2.5 Consensus Voting

An image is flagged as anomalous if it is identified by **at least 2 of 3** independent methods. This avoids reliance on any single threshold estimate and provides robust anomaly detection.

In [ ]:
# Consensus threshold (≥2 of 3 methods must agree)
is_anomaly, effective_threshold, all_info = consensus_threshold(
    novelty_scores, min_votes=2
)

n_anomalies = is_anomaly.sum()
print(f'\nFinal anomaly count: {n_anomalies}')
print(f'Effective threshold: {effective_threshold:.6f}')
print(f'Anomaly rate: {n_anomalies / len(novelty_scores) * 100:.2f}%')

In [ ]:
# Visualize score distribution with all threshold lines
plot_score_distribution(
    novelty_scores,
    thresholds={
        'GPD (EVT)': threshold_gpd,
        'KDE Knee': threshold_kde,
        'MAD (3.5σ)': threshold_mad,
    },
    title=f'Novelty Score Distribution ({BEST_VERSION.upper()})',
    save_path=os.path.join(PLOTS_DIR, f'score_distribution_{BEST_VERSION}.png'),
)

## 2.6 Latent Space Colored by Novelty Score

In [ ]:
# Re-plot t-SNE and UMAP colored by novelty scores
plot_tsne(
    latents, perplexity=30,
    title=f'{BEST_VERSION.upper()} — t-SNE (colored by Novelty Score)',
    scores=novelty_scores,
    save_path=os.path.join(PLOTS_DIR, f'tsne_scored_{BEST_VERSION}.png'),
)

plot_umap(
    latents, n_neighbors=15, min_dist=0.1,
    title=f'{BEST_VERSION.upper()} — UMAP (colored by Novelty Score)',
    scores=novelty_scores,
    save_path=os.path.join(PLOTS_DIR, f'umap_scored_{BEST_VERSION}.png'),
)

## 2.7 Save Scores & Metadata

In [ ]:
# Save all scores to CSV
scores_df = save_scores(
    scores=novelty_scores,
    filenames=filenames,
    is_anomaly=is_anomaly,
    info_df=info_df,
    save_path=os.path.join(SCORES_DIR, f'novelty_scores_{BEST_VERSION}.csv'),
    version=BEST_VERSION,
)

# Show top anomalies
print('\nTop 10 Most Anomalous Images:')
print('=' * 70)
print(scores_df.head(10).to_string(index=False))

In [ ]:
# Location analysis: if metadata is available, show geographical distribution
if 'latitude' in scores_df.columns and 'longitude' in scores_df.columns:
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Normal images
    normal = scores_df[~scores_df['is_anomaly']]
    anomalous = scores_df[scores_df['is_anomaly']]
    
    ax.scatter(normal['longitude'], normal['latitude'],
               c='#3498db', s=5, alpha=0.3, label='Normal')
    ax.scatter(anomalous['longitude'], anomalous['latitude'],
               c='#e74c3c', s=50, alpha=0.9, marker='*',
               edgecolors='black', linewidth=0.5, label='Anomalous')
    
    ax.set_xlabel('Longitude (°)', fontsize=12)
    ax.set_ylabel('Latitude (°)', fontsize=12)
    ax.set_title('Anomaly Locations on Mars', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'anomaly_locations.png'), dpi=150)
    plt.show()
else:
    print('Latitude/longitude not available in metadata.')

print('\n✓ Phase 2 complete. Scores saved for Phase 3.')